In this notebook I will take in my basefile (with strike treatment and covariates added)

create a train/test split 

and explore some meta learners

In [2]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML

e:\tfl_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here I bring in my basefile which is saved as a parquet.

This basefile contains London TFL bike station usage data per station per hour over a three period. This is my target or Y. This data is from tfl free to download data

Joined onto this basefile is a selection of covariates, of which you would expect to have some impact on the the bike usage each hour. This is my (observed confounders) covariate data or X. This is from time based data and weather sources.

Next we have a column with if that station at that hour had a line serve it that a strike on that day. This is my treatment column or T.

I am interested in the causal inference question: What is the effect of strike days on the usage of TFL bikes?

I calculate the CATE (conditional average treatment effect) using several META learner methods.

In [ ]:
PATH = "base_station_hour_coords_with_covariates.parquet"  # adjust

df = pd.read_parquet(PATH)

df.size
df.columns

(9197640, 26)


Index(['station_id', 'trips_start', 'date', 'ts', 'y_log1p', 'strike_exposed',
       'lat', 'lon', 'station_name', 'weather_cell_id', 'hour', 'dow', 'month',
       'year', 'doy', 'is_weekend', 'is_am_peak', 'is_pm_peak',
       'is_bank_holiday', 'temperature_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m',
       'weather_code'],
      dtype='str')

In [4]:
df.head()


,station_id,trips_start,date,ts,y_log1p,strike_exposed,lat,lon,station_name,weather_cell_id,...,is_am_peak,is_pm_peak,is_bank_holiday,temperature_2m,relative_humidity_2m,precipitation,rain,cloud_cover,wind_speed_10m,weather_code
0,1,2016-01-10 09:00:00,2016-01-10,4,1.609438,0,51.529163,-0.10997,"River Street , Clerkenwell",g1.100_lat5214_lon-7,...,1,0,0,5.7,84.0,0.0,0.0,100.0,18.7,3.0
1,1,2016-01-10 10:00:00,2016-01-10,1,0.693147,0,51.529163,-0.10997,"River Street , Clerkenwell",g1.100_lat5214_lon-7,...,0,0,0,6.2,83.0,0.0,0.0,92.0,16.0,3.0
2,1,2016-01-10 11:00:00,2016-01-10,2,1.098612,0,51.529163,-0.10997,"River Street , Clerkenwell",g1.100_lat5214_lon-7,...,0,0,0,6.9,81.0,0.0,0.0,75.0,17.2,2.0
3,1,2016-01-10 12:00:00,2016-01-10,2,1.098612,0,51.529163,-0.10997,"River Street , Clerkenwell",g1.100_lat5214_lon-7,...,0,0,0,7.5,74.0,0.0,0.0,88.0,21.6,3.0
4,1,2016-01-10 13:00:00,2016-01-10,2,1.098612,0,51.529163,-0.10997,"River Street , Clerkenwell",g1.100_lat5214_lon-7,...,0,0,0,7.7,72.0,0.0,0.0,82.0,21.8,3.0


In [5]:
COL_TIME = "trips_start"
COL_Y    = "ts"
COL_T    = "strike_exposed"

df[COL_TIME] = pd.to_datetime(df[COL_TIME])
df[COL_Y] = pd.to_numeric(df[COL_Y], errors="coerce")
df[COL_T] = pd.to_numeric(df[COL_T], errors="coerce").fillna(0).astype(int)

df = df.dropna(subset=[COL_TIME, COL_Y])
df["y_log1p"] = np.log1p(df[COL_Y].astype(float))

df[[COL_TIME, COL_Y, COL_T, "y_log1p"]].head()

,trips_start,ts,strike_exposed,y_log1p
0,2016-01-10 09:00:00,4,0,1.609438
1,2016-01-10 10:00:00,1,0,0.693147
2,2016-01-10 11:00:00,2,0,1.098612
3,2016-01-10 12:00:00,2,0,1.098612
4,2016-01-10 13:00:00,2,0,1.098612


We apply basic feature transformations and use a time based modelling split

In [6]:
SPLIT_DATE = "2018-01-01"

train = df[df[COL_TIME] < SPLIT_DATE].copy()
test  = df[df[COL_TIME] >= SPLIT_DATE].copy()

print("train:", train.shape, "test:", test.shape)

Y_train = train["y_log1p"].values
T_train = train[COL_T].values.astype(int)

Y_test = test["y_log1p"].values
T_test = test[COL_T].values.astype(int)

train: (6072792, 26) test: (3124848, 26)


In [8]:
# Candidate categorical features
#cat_cols = ["station_id"]  # add "weather_cell_id" if you want

# Candidate numeric features (only keep those that exist)
num_cols = [
    "hour", 
    "dow",
    "month",
    "year",
    "doy",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "is_weekend",
    "is_am_peak",
    "is_pm_peak",
    "is_bank_holiday",
    "lat",
    "lon",
]

cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in num_cols if c in train.columns]

print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

pre = ColumnTransformer(
    transformers=[
      #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

Xtr = pre.fit_transform(train[num_cols])
Xte = pre.transform(test[num_cols])

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

cat_cols: ['station_id']
num_cols: ['hour', 'dow', 'month', 'year', 'doy', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'is_weekend', 'is_am_peak', 'is_pm_peak', 'is_bank_holiday', 'lat', 'lon']
Xtr: (6072792, 17) Xte: (3124848, 17)


Instantiate two Random Forest items - Regressor and Classifier

In [9]:
rf_y = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=100,
    n_jobs=-1,
    random_state=0,
)

rf_t = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=100,
    n_jobs=-1,
    random_state=0,
)

Finally I look at training meta learners using my basefile and random forest objects

We train a S,T and X learner.

These all show different levels of CATE, with the T learner actually showing a negative effect.

In [11]:
import time

start = time.time()

s = SLearner(overall_model=rf_y)
s.fit(Y_train, T_train, X=Xtr)
tau_s = s.effect(Xte)

print("SLearner done in", time.time() - start, "seconds")

Slearner_end = time.time()

t = TLearner(models=RandomForestRegressor(
    n_estimators=200, min_samples_leaf=100, n_jobs=-1, random_state=0
))
t.fit(Y_train, T_train, X=Xtr)
tau_t = t.effect(Xte)

print("TLearner done in", time.time() - Slearner_end, "seconds")

Tlearner_end = time.time()

x = XLearner(models=RandomForestRegressor(
    n_estimators=200, min_samples_leaf=100, n_jobs=-1, random_state=0
))
x.fit(Y_train, T_train, X=Xtr)
tau_x = x.effect(Xte)

print("XLearner done in", time.time() - Tlearner_end, "seconds")

print("ATE S:", float(np.mean(tau_s)))
print("ATE T:", float(np.mean(tau_t)))
print("ATE X:", float(np.mean(tau_x)))

e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


SLearner done in 5847.690194606781 seconds


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TLearner done in 3405.2842302322388 seconds


e:\tfl_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


XLearner done in 6843.624732494354 seconds
ATE S: 0.0011210937509432784
ATE T: -0.02259218853499545
ATE X: 0.0016370561966232162
